In [64]:
import pandas as pd
import numpy as np
import os
import joblib
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Dense, Concatenate, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings('ignore')

In [65]:
# --- 定数とパスの設定 ---
RANDOM_STATE = 42
TEST_SIZE = 0.2
DATA_DIR = '../data/processed/'
MODEL_DIR = '../models/'
PROCESSED_FILE = 'features.parquet'
PROCESSED_PATH = os.path.join(DATA_DIR, PROCESSED_FILE)

# 保存ファイル名
MODEL_FILE_XGB_AD = 'xgb_model_ad.pkl'
MODEL_FILE_NN_EMB_E = 'nn_model_e.h5'
MODEL_FILE_PREP_E = 'e_segment_preprocessing.pkl'
MODEL_FILE_FIXED_BC = 'fixed_pred_bc.pkl'
MODEL_FILE_MAE_AE = 'mae_ae_threshold.pkl' # A/E分離のための基準値（特徴量ルールが見つからない場合の代替案）

# XGBoost の設定を修正
XGB_CONFIG = {
    'enable_categorical': True, 
    'objective': 'reg:squarederror', 
    'n_estimators': 1000,
    'learning_rate': 0.05, 
    'max_depth': 6, 
    'subsample': 0.8, 
    'colsample_bytree': 0.8,
    'random_state': RANDOM_STATE, 
    'n_jobs': -1, 
    # 🚨 ここを削除またはコメントアウト
    # 'early_stopping_rounds': 50, 
    'missing': np.nan
}
TARGET_COL = '取引価格（総額）_log'

os.makedirs(MODEL_DIR, exist_ok=True)

In [66]:
# --- 1. データロードと初期処理 (セル 1 の前準備) ---
df = pd.read_parquet(PROCESSED_PATH)
df.columns

Index(['市区町村コード', '都道府県名', '市区町村名', '地区名', '最寄駅：名称', '最寄駅：距離（分）', '間取り',
       '面積（㎡）', '建築年', '建物の構造', '用途', '今後の利用目的', '都市計画', '建ぺい率（％）', '容積率（％）',
       '取引時点', '取引価格（総額）_log', '取引時点_年', '建築年_西暦', '築年数_欠損', '取引時点での築年数',
       '取引の事情等_調停・競売等', '取引の事情等_関係者間取引', '取引の事情等_その他', '改装_改装済', '改装_未改装',
       '間取り_grouped_その他', '間取り_grouped_オープンフロア', '間取り_grouped_欠損値',
       '間取り_grouped_１ＤＫ', '間取り_grouped_１Ｋ', '間取り_grouped_１ＬＤＫ',
       '間取り_grouped_１Ｒ', '間取り_grouped_２ＤＫ', '間取り_grouped_２Ｋ',
       '間取り_grouped_２ＬＤＫ', '間取り_grouped_２ＬＤＫ＋Ｓ', '間取り_grouped_３ＤＫ',
       '間取り_grouped_３ＬＤＫ', '間取り_grouped_４ＤＫ', '間取り_grouped_４ＬＤＫ', '都市計画_高価格帯',
       '都市計画_中価格帯', '都市計画_低価格帯', '人口密度', '市区町村人口密度', '犯罪発生率', '築年数_2乗',
       '築年数_3乗', '築年数_log', '面積_log', '面積_平方根', '築年数×面積', '築年数×駅距離',
       '築年数×建ぺい率', '築年数×容積率', '築年数×人口密度', '面積×駅距離', '面積×建ぺい率', '面積×容積率',
       '面積×人口密度', '建築可能性', '容積率_建ぺい率比', '建ぺい率_2乗', '容積率_2乗', '駅距離_逆数',
       '駅距離_log', '駅距離_2乗', '駅距離×建ぺい率', '駅距離×容積率', '人口密度_log', '市区町村人口密度_l

In [67]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 566680 entries, 0 to 566679
Data columns (total 83 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   市区町村コード              566680 non-null  int64  
 1   都道府県名                566680 non-null  object 
 2   市区町村名                566680 non-null  object 
 3   地区名                  566463 non-null  object 
 4   最寄駅：名称               566680 non-null  object 
 5   最寄駅：距離（分）            566680 non-null  float64
 6   間取り                  547938 non-null  object 
 7   面積（㎡）                566680 non-null  int64  
 8   建築年                  554195 non-null  object 
 9   建物の構造                558786 non-null  object 
 10  用途                   524906 non-null  object 
 11  今後の利用目的              254799 non-null  object 
 12  都市計画                 566572 non-null  object 
 13  建ぺい率（％）              566680 non-null  float64
 14  容積率（％）               566680 non-null  float64
 15  取引時点             

In [68]:
drop_cols = ['市区町村コード', '間取り','用途', '今後の利用目的', '都市計画', '建築年', '取引時点',
             '取引の事情等_その他','改装_未改装', '間取り_grouped_その他',TARGET_COL]
X_cols = [col for col in df.columns if col not in drop_cols]
X_full = df[X_cols].copy()
Y_full = df[TARGET_COL].copy()

categorical_cols_common = ['都道府県名', '市区町村名', '地区名', '最寄駅：名称', '建物の構造']
for col in categorical_cols_common:
    if col in X_full.columns:
        X_full[col] = X_full[col].astype('category')

numerical_cols_all = X_full.select_dtypes(include=[np.number]).columns
X_full[numerical_cols_all] = X_full[numerical_cols_all].replace([np.inf, -np.inf], np.nan)
X_full[numerical_cols_all] = X_full[numerical_cols_all].fillna(X_full[numerical_cols_all].median())

In [69]:
# --- 1-1. B/C セグメントの選定 ---
print("--- Step 1-1: B (特殊ノイズ) および C (困難構造) の選定 ---")
# df_segmentation が df のコピーとして初期化されていることを前提とします
df_segmentation = df.copy() 
df_segmentation['Segment_Type'] = 'None'
transaction_cols = [col for col in X_full.columns if col.startswith('取引の事情等')]

# df_segmentationに取引の事情等カラムをコピー
for col in transaction_cols:
    if col in X_full.columns:
        df_segmentation[col] = X_full[col] 

# B: 特殊ノイズのマスク定義 (取引の事情等)
shijo_cols_B = ['取引の事情等_調停・競売等', '取引の事情等_関係者間取引', '取引の事情等_他の権利・負担付き',
                '取引の事情等_他の権利・負担付き、調停・競売等', '取引の事情等_調停・競売等、瑕疵有りの可能性']
shijo_cols_B = [col for col in shijo_cols_B if col in df_segmentation.columns]

if shijo_cols_B:
    mask_B = df_segmentation[shijo_cols_B].any(axis=1)
    df_segmentation.loc[mask_B, 'Segment_Type'] = 'B:特殊ノイズ'
    print(f"   - Bセグメント選定完了: {mask_B.sum():,}件")
else:
    mask_B = pd.Series(False, index=df_segmentation.index)
    print("   - Bセグメント選定スキップ: 取引の事情等カラムが見つかりません。")

# C: 困難構造のマスク定義
mask_C = ((df_segmentation['建物の構造'] == '軽量鉄骨造') | (df_segmentation['建物の構造'] == 'ブロック造') | (df_segmentation['建物の構造'] == 'ＲＣ、木造'))
df_segmentation.loc[mask_C & (df_segmentation['Segment_Type'] == 'None'), 'Segment_Type'] = 'C:困難構造'
print(f"   - Cセグメント選定完了: {(df_segmentation['Segment_Type'] == 'C:困難構造').sum():,}件")

--- Step 1-1: B (特殊ノイズ) および C (困難構造) の選定 ---
   - Bセグメント選定完了: 14,852件
   - Cセグメント選定完了: 36件


In [70]:
# --- 1-2. D (プレミア/希少) の選定 ---
print("\n--- Step 1-2: D (プレミア/希少) の選定 ---")
# 地区名_頻度_log のisnull()は特徴量エンジニアリングが完了している前提
mask_D = (((df_segmentation['都道府県名'] == '東京都') & (df_segmentation['取引時点での築年数'] < 5)) | (df_segmentation['地区名_頻度_log'].isnull()))
df_segmentation.loc[mask_D & (df_segmentation['Segment_Type'] == 'None'), 'Segment_Type'] = 'D:プレミア/希少'
print(f"   - Dセグメント選定完了: {(df_segmentation['Segment_Type'] == 'D:プレミア/希少').sum():,}件")


--- Step 1-2: D (プレミア/希少) の選定 ---
   - Dセグメント選定完了: 30,066件


In [71]:
# --- Step 2-0: 初期 XGBoost モデルによる訓練データ予測の実行 ---
print("\n--- Step 2-0: 初期 XGBoost モデルによる訓練データ予測の実行 ---")
from xgboost import XGBRegressor # 既にインポート済みかもしれませんが念のため

# 1. 初期 XGBoost モデルをインスタンス化
# 🚨 XGB_CONFIG は前のセルで定義済みと仮定
initial_xgb_model = XGBRegressor(**XGB_CONFIG)

# 2. 訓練データ全体 (X_full/X_train_full) を使用してモデルを訓練
print("    - 初期モデル (XGBoost) の訓練を開始します...")
# X_fullは前処理済みでカテゴリ型が適切に設定されている前提
# X_train_full は X_full と同じインスタンスとして使用
initial_xgb_model.fit(X_full, Y_full) 
print("    - ✅ 初期モデルの訓練完了。")

# 3. 訓練したモデルを使用して、訓練データ全体に対する予測を実行
# y_pred_initial_model に代入 (MAE計算に使用)
y_pred_initial_model = pd.Series(
    initial_xgb_model.predict(X_full), 
    index=X_full.index
)
print(f"    - ✅ 訓練済み初期モデルによる予測完了。件数: {len(y_pred_initial_model):,}件")


# --------------------------------------------------------------------------
# --- 2-0-1. 訓練済み要素の保存 (推論に必要なパラメータ) ---
# --------------------------------------------------------------------------
print("\n--- Step 2-0-1: 訓練済み要素の保存 (prediction_params.pkl) ---")

# 予測に必要なパラメータ辞書を作成/更新
try:
    prediction_params = joblib.load(os.path.join(MODEL_DIR, 'prediction_params.pkl'))
except FileNotFoundError:
    prediction_params = {}

# 1. ノイズ分類器/XGBRegressorの入力特徴量リスト
prediction_params['noise_features'] = X_full.columns.tolist()

# 2. 数値特徴量の中央値 (NaN補完用)
# 🚨 MEDIAN_VALUES は前の前処理で計算された値を使用
MEDIAN_VALUES = X_full[numerical_cols_all].median().to_dict()
prediction_params['median_values'] = MEDIAN_VALUES

# 3. Eセグメント用の数値/カテゴリ特徴量リスト (NNモデルの入力用)
# 🚨 NUMERIC_FEATURES, CATEGORICAL_FEATURES が定義されていない場合、ここで定義
if 'NUMERIC_FEATURES' not in locals() or 'CATEGORICAL_FEATURES' not in locals():
    NUMERIC_FEATURES = X_full.select_dtypes(include=np.number).columns.tolist()
    CATEGORICAL_FEATURES = X_full.select_dtypes(include='category').columns.tolist()

prediction_params['numeric_features'] = NUMERIC_FEATURES
prediction_params['categorical_features'] = CATEGORICAL_FEATURES

# 4. B/Cセグメントの予測値 (後の推論で利用)
# ルールベースで確定したB/CセグメントのY_full平均値を採用
mask_bc = df_segmentation['Segment_Type'].isin(['B:特殊ノイズ', 'C:困難構造'])
bc_average_log = Y_full.loc[mask_bc].mean()
prediction_params['bc_average_log'] = bc_average_log


# prediction_params.pkl の保存
joblib.dump(prediction_params, os.path.join(MODEL_DIR, 'prediction_params.pkl'))
print(f"    - ✅ prediction_params.pkl を保存しました。")
print(f"    - 💡 B/Cセグメント平均予測値 (log): {bc_average_log:.6f}")
print("-" * 50)


--- Step 2-0: 初期 XGBoost モデルによる訓練データ予測の実行 ---
    - 初期モデル (XGBoost) の訓練を開始します...
    - ✅ 初期モデルの訓練完了。
    - ✅ 訓練済み初期モデルによる予測完了。件数: 566,680件

--- Step 2-0-1: 訓練済み要素の保存 (prediction_params.pkl) ---
    - ✅ prediction_params.pkl を保存しました。
    - 💡 B/Cセグメント平均予測値 (log): 6.999520
--------------------------------------------------


In [72]:
y_pred_initial_model

0         7.124063
1         6.449182
2         6.980862
3         7.531829
4         6.870093
            ...   
566675    7.519138
566676    6.335159
566677    7.297294
566678    7.345521
566679    7.554316
Length: 566680, dtype: float32

In [73]:
# --- 2-1. A/E 分離用二値分類器の訓練データ準備 (MAEパーセンタイル抽出) ---
print("\n--- Step 2-1: MAE パーセンタイルによる A/E 分離用訓練データ準備 ---")
from xgboost import XGBClassifier

# 1. A/E 候補となるデータの抽出
train_indices = X_train_full.index.intersection(df_segmentation.index)
mask_ae_candidate_train = (df_segmentation['Segment_Type'] == 'None').loc[train_indices]

# A/E 候補のデータ抽出
X_train_ae = X_train_full.loc[mask_ae_candidate_train]
y_train_ae = Y_train_full.loc[mask_ae_candidate_train]
y_pred_ae_initial = y_pred_initial_model.loc[X_train_ae.index] 

if len(X_train_ae) == 0:
    print("    - ❌ エラー: A/E候補データ ('None'セグメント) が0件です。データ抽出ロジックを確認してください。")
    raise RuntimeError("A/E候補データが0件")

# 2. 🚨 目的変数の作成: MAEのパーセンタイルに基づき E (1) をラベリング
mae_series = np.abs(y_train_ae - y_pred_ae_initial)

# 🚨 上位 1.0% (99.0パーセンタイル) を E セグメントとする動的な閾値を設定
e_segment_percentile = 99.0 
mae_threshold_for_binary = np.percentile(mae_series, e_segment_percentile)

print(f"    - 💡 MAE {e_segment_percentile:.1f}パーセンタイル閾値: {mae_threshold_for_binary:.6f}")

# MAEが動的閾値を超えるものを E (1) とする
y_ae_binary = (mae_series > mae_threshold_for_binary).astype(int)

# 訓練データと目的変数を準備
X_train_binary = X_train_ae.copy()
y_train_binary = y_ae_binary  # 🚨 A/Eの二値ラベルを格納

# Eセグメント件数の確認
count_e = y_train_binary.sum()
count_a = len(y_train_binary) - count_e

print(f"    - Eセグメント (1) ラベル件数: {count_e:,}件 (割合: {y_train_binary.mean()*100:.2f}%)")
print("-" * 50)


--- Step 2-1: MAE パーセンタイルによる A/E 分離用訓練データ準備 ---
    - 💡 MAE 99.0パーセンタイル閾値: 0.366070
    - Eセグメント (1) ラベル件数: 5,218件 (割合: 1.00%)
--------------------------------------------------


In [74]:
# --------------------------------------------------------------------------
# --- 2-2. XGBoost分類器の訓練 ---
# --------------------------------------------------------------------------
print("\n--- Step 2-2: A vs E 二値分類器 (XGBoost) の訓練 ---")

if count_e == 0:
    print("    - ❌ 訓練失敗: Eセグメントが0件のため分類器を訓練できません。")
    raise RuntimeError("Eセグメント件数が0件のため訓練を中止。")

# 🚨 不均衡データ対策: クラス重み付けを計算
scale_pos_weight_value = count_a / count_e
print(f"    - 💡 scale_pos_weight を {scale_pos_weight_value:.2f} に設定します。")

# モデルの訓練 (ハイパーパラメータ調整と重み付けを適用)
# 🚨 RANDOM_STATE は事前に定義されているものと仮定
classifier_ae = XGBClassifier(
    enable_categorical=True, 
    objective='binary:logistic',
    eval_metric='logloss',
    use_label_encoder=False, 
    n_estimators=300, 
    learning_rate=0.05, 
    max_depth=7, 
    scale_pos_weight=scale_pos_weight_value, # 🚨 重み付けを適用
    random_state=RANDOM_STATE
)

# 訓練実行
classifier_ae.fit(X_train_binary, y_train_binary)
print("    - ✅ A vs E 二値分類器 (XGBoost) の訓練完了。")
print("-" * 50)


--- Step 2-2: A vs E 二値分類器 (XGBoost) の訓練 ---
    - 💡 scale_pos_weight を 98.99 に設定します。
    - ✅ A vs E 二値分類器 (XGBoost) の訓練完了。
--------------------------------------------------


In [75]:
# --------------------------------------------------------------------------
# --- 2-3. A/E 分離器による分類結果の適用 (モデル適用ベースの件数調整版) ---
# --------------------------------------------------------------------------
print("\n--- Step 2-3: A/E 分離器による分類結果の適用 (モデル適用ベースの件数調整版) ---")
import joblib
import os

# 🚨 前ステップで定義された変数を使用: 
# classifier_ae, X_train_binary, train_indices, df_segmentation, mae_threshold_for_binary
# count_e (Step 2-1で定義された訓練時のEセグメント件数)

# 1. 分類器による確率予測実行 (A/E候補のみ)
y_pred_proba_e = classifier_ae.predict_proba(X_train_binary)[:, 1] 
y_pred_proba_series = pd.Series(y_pred_proba_e, index=X_train_binary.index)

# 2. 予測確率に基づき、訓練時の E セグメント件数と**同数**になるように閾値を設定する
# 💡 これが「モデルの分類結果を適用しつつ、E件数を確保する」ための代替策です
# 訓練時 E件数 (count_e) に基づくパーセンタイルを計算
e_count_percent = (1 - (count_e / len(X_train_binary))) * 100
proba_threshold_adjusted = np.percentile(y_pred_proba_e, e_count_percent)

print(f"    - 💡 訓練時の E ラベル件数 ({count_e:,}件) に基づく確率閾値: {proba_threshold_adjusted:.4f}")

# 3. 予測確率が調整後の閾値を超えたデータのインデックスを取得
e_indices_predicted = y_pred_proba_series[y_pred_proba_series > proba_threshold_adjusted].index

# 4. df_segmentation の 'None' セグメントを 'E:高誤差ノイズ' に更新
# E に分類されたインデックスは、元の 'None' セグメントから抽出されたもの
df_segmentation.loc[e_indices_predicted, 'Segment_Type'] = 'E:高誤差ノイズ'

# 5. 残った 'None' のデータを 'A' に更新
# 修正: 元々 'None' であったデータのうち、E に分類されなかったものを 'A' にする
# df_segmentation が 'None' のままで残っている行が、最終的な A セグメント候補
mask_to_update_a = df_segmentation['Segment_Type'] == 'None'
df_segmentation.loc[mask_to_update_a, 'Segment_Type'] = 'A:高精度'


# 6. 結果確認とパラメータの最終保存
count_e_final = (df_segmentation['Segment_Type'] == 'E:高誤差ノイズ').sum()
count_a_final = (df_segmentation['Segment_Type'] == 'A:高精度').sum()

print(f"    - ✅ df_segmentation を最終更新完了。")
print(f"    - 💡 最終的な A セグメント件数: {count_a_final:,}件")
print(f"    - 💡 最終的な E セグメント件数: {count_e_final:,}件") 

# 7. MAE閾値と予測確率閾値の保存
# prediction_params.pkl は既に存在し、MODEL_DIRは定義済みと仮定
prediction_params = joblib.load(os.path.join(MODEL_DIR, 'prediction_params.pkl'))
prediction_params['mae_ae_threshold'] = mae_threshold_for_binary
prediction_params['proba_e_threshold'] = proba_threshold_adjusted # 新しい確率閾値を保存
joblib.dump(prediction_params, os.path.join(MODEL_DIR, 'prediction_params.pkl'))
print(f"    - ✅ A/E分離閾値 ({mae_threshold_for_binary:.6f}) と確率閾値を prediction_params.pkl に保存。")
print("-" * 50)


--- Step 2-3: A/E 分離器による分類結果の適用 (モデル適用ベースの件数調整版) ---
    - 💡 訓練時の E ラベル件数 (5,218件) に基づく確率閾値: 0.8379
    - ✅ df_segmentation を最終更新完了。
    - 💡 最終的な A セグメント件数: 516,508件
    - 💡 最終的な E セグメント件数: 5,218件
    - ✅ A/E分離閾値 (0.366070) と確率閾値を prediction_params.pkl に保存。
--------------------------------------------------


In [76]:
# --- 3-3. A/D セグメント専用 XGBoost モデルの訓練 (X_train_e 抽出ロジックを追加) ---
print("\n--- Step 3-3: A/D セグメント専用 XGBoost モデルの訓練 ---")
from xgboost import XGBRegressor

# 1. A/D セグメントを抽出
train_indices = X_train_full.index.intersection(df_segmentation.index)
mask_ad_train_final = df_segmentation.loc[train_indices, 'Segment_Type'].isin(['A:高精度', 'D:プレミア/希少'])

X_train_ad_final = X_train_full.loc[mask_ad_train_final]
Y_train_ad_final = Y_train_full.loc[mask_ad_train_final]

if len(X_train_ad_final) == 0:
    print("   - ❌ エラー: A/D セグメントの訓練データが0件のため訓練をスキップしました。")
    raise RuntimeError("A/D訓練データが0件")
    
print(f"   - 💡 A/D セグメント訓練データ件数: {len(X_train_ad_final):,}件")


# 2. XGBoostモデルの定義と訓練
model_a_new = XGBRegressor(
    objective='reg:squarederror',
    enable_categorical=True,
    n_estimators=500,        
    learning_rate=0.03,      
    max_depth=8,             
    n_jobs=-1,
    random_state=RANDOM_STATE
)
model_a_new.fit(X_train_ad_final, Y_train_ad_final)
print("   - ✅ A/D セグメント専用 XGBoost モデル (model_a_new) の訓練完了。")


# 3. 🚨 Eセグメントの予測結果 (y_pred_e_train) の初期化
# Eセグメントのデータをここで再抽出します
mask_e_train = df_segmentation.loc[train_indices, 'Segment_Type'] == 'E:高誤差ノイズ'
X_train_e = X_train_full.loc[mask_e_train] # 🚨 X_train_e を再定義

try:
    # y_pred_e_train が定義済みであればそのまま
    _ = y_pred_e_train
except NameError:
    # 代わりに初期モデルの予測結果を使用（この後の Step 3-2 で使用されます）
    # X_train_e のインデックスを使って y_pred_initial_model から抽出
    y_pred_e_train = y_pred_initial_model.loc[X_train_e.index] 
    print("   - ℹ️ Eセグメント予測 y_pred_e_train は未定義のため、初期モデル予測で初期化しました。")


--- Step 3-3: A/D セグメント専用 XGBoost モデルの訓練 ---
   - 💡 A/D セグメント訓練データ件数: 546,574件
   - ✅ A/D セグメント専用 XGBoost モデル (model_a_new) の訓練完了。


In [77]:
!pip install tensorflow

In [78]:
# --- 3-5. E セグメント専用 NN モデルの訓練と予測 (NN使用 - 埋め込み版 - 最終修正) ---
print("\n--- Step 3-5: E セグメント専用 NN モデルの訓練と予測 (NN使用 - 埋め込み版 - 最終修正) ---")
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Normalization, StringLookup, Embedding, Flatten
import numpy as np
import pandas as pd

# 1. E セグメントのデータ抽出 (再掲)
train_indices = X_train_full.index.intersection(df_segmentation.index)
mask_e_train_final = df_segmentation.loc[train_indices, 'Segment_Type'] == 'E:高誤差ノイズ'

X_train_e_final = X_train_full.loc[mask_e_train_final]
Y_train_e_final = Y_train_full.loc[mask_e_train_final]

if len(X_train_e_final) == 0:
    print("   - ❌ エラー: E セグメントの訓練データが0件です。")
    raise RuntimeError("E訓練データが0件")

print(f"   - 💡 E セグメント訓練データ件数: {len(X_train_e_final):,}件")


# --- 2. Kerasのプリプロセッシングレイヤーによる特徴量処理の定義 ---

NUMERIC_FEATURES = X_train_e_final.select_dtypes(include=np.number).columns.tolist()
CATEGORICAL_FEATURES = X_train_e_final.select_dtypes(include=['object', 'category']).columns.tolist()

all_inputs = []
encoded_features = []

# 2.1. 数値特徴量の処理 (Normalization)
for name in NUMERIC_FEATURES:
    col = X_train_e_final[name].astype('float32')
    col_2d = col.values.reshape(-1, 1)
    
    input_ = keras.Input(shape=(1,), name=name)
    
    normalization_layer = Normalization()
    normalization_layer.adapt(col_2d)
    
    encoded_feature = normalization_layer(input_)
    all_inputs.append(input_)
    encoded_features.append(encoded_feature)

# 2.2. カテゴリカル特徴量の処理 (StringLookup + Embedding)
for name in CATEGORICAL_FEATURES:
    col = X_train_e_final[name]
    
    if pd.api.types.is_categorical_dtype(col.dtype):
        col = col.astype(object)
        
    # NaNを 'missing' という文字列に置換
    col_filled = col.fillna('missing').astype(str)
    vocab = np.array(col_filled.unique(), dtype=object) 
    
    # 💡 StringLookupの仕様調整
    # StringLookupはデフォルトで index 0を OOV に、index 1を Masking に割り当てるため、
    # 語彙サイズは len(vocab) + 2 となり、Embedding層のインデックス範囲外になる
    # 🚨 修正: mask_token=None, oov_token="OOV" (デフォルトを許容) にして、語彙サイズを正しく計算する
    # Embedding層の input_dim は語彙数 + 1 (OOVトークンのため)
    
    vocab_size = len(vocab) + 1 # 語彙数 + OOVトークン (デフォルト: index 0)
    EMBEDDING_DIM = 5 if vocab_size > 5 else 2

    input_ = keras.Input(shape=(1,), dtype='string', name=name)
    
    # 🚨 修正点: mask_token=None (マスキングインデックスを使わない) に設定
    lookup_layer = StringLookup(vocabulary=vocab, mask_token=None, num_oov_indices=1) 
    encoded_integer = lookup_layer(input_) # 整数インデックス (0 から vocab_size-1)

    # 整数インデックスを埋め込みベクトルに変換
    embedding_layer = Embedding(input_dim=vocab_size, output_dim=EMBEDDING_DIM)(encoded_integer)
    
    encoded_feature = Flatten()(embedding_layer)
    
    all_inputs.append(input_)
    encoded_features.append(encoded_feature)

# 3. NNモデルの定義とコンパイル
x = keras.layers.concatenate(encoded_features)
x = keras.layers.Dense(64, activation="relu")(x)
x = keras.layers.Dropout(0.2)(x)
x = keras.layers.Dense(32, activation="relu")(x)
output = keras.layers.Dense(1, name="price_prediction")(x)

model_e_nn = keras.Model(all_inputs, output)

# モデルのコンパイルと訓練
model_e_nn.compile(optimizer='adam', loss='mae', metrics=['mae'])

# 4. Keras ModelにDataFrameを渡すための入力辞書を作成
X_train_e_dict = {}
for name in NUMERIC_FEATURES:
    X_train_e_dict[name] = np.array(X_train_e_final[name])
for name in CATEGORICAL_FEATURES:
    col = X_train_e_final[name]
    if pd.api.types.is_categorical_dtype(col.dtype):
        col = col.astype(object)
    
    # fillnaした文字列配列を渡す（訓練時と同じ前処理）
    X_train_e_dict[name] = col.fillna('missing').values 

history_e = model_e_nn.fit(
    X_train_e_dict, 
    Y_train_e_final.values, 
    epochs=50, 
    batch_size=64, 
    verbose=0, 
    validation_split=0.1
)
print("   - ✅ E セグメント専用 NN モデルの訓練完了。")


# 5. 訓練データに対する予測の実行
y_pred_e_train_array = model_e_nn.predict(X_train_e_dict).flatten()

# 予測結果をPandas Seriesに格納し直す (Step 3-2で使えるように更新)
y_pred_e_train = pd.Series(y_pred_e_train_array, index=X_train_e_final.index) 
print("   - ✅ E セグメント予測結果 (y_pred_e_train) の更新完了。")


--- Step 3-5: E セグメント専用 NN モデルの訓練と予測 (NN使用 - 埋め込み版 - 最終修正) ---
   - 💡 E セグメント訓練データ件数: 5,218件
   - ✅ E セグメント専用 NN モデルの訓練完了。
164/164 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step
   - ✅ E セグメント予測結果 (y_pred_e_train) の更新完了。


In [79]:
# --- 3-2. A/D (model_a_new) と E (NN予測) の訓練データ適用と統合 ---
print("\n--- Step 3-2: A/D, E への訓練データ適用と予測統合 (最終評価) ---")
from sklearn.metrics import mean_absolute_error

# 1. 最終予測配列の準備 (インデックスは訓練データを使用)
y_pred_train_final = pd.Series(index=Y_train_full.index, dtype=float)

# 2. A/Dセグメントの予測 (新モデル model_a_new を使用)
train_indices = Y_train_full.index.intersection(df_segmentation.index)
mask_ad_train = df_segmentation.loc[train_indices, 'Segment_Type'].isin(['A:高精度', 'D:プレミア/希少'])
X_train_ad = X_train_full[mask_ad_train]

if len(X_train_ad) > 0:
    # 訓練済みの model_a_new を使用
    y_pred_ad_log = model_a_new.predict(X_train_ad).flatten() 
    y_pred_train_final.loc[X_train_ad.index] = y_pred_ad_log
    print(f"   - ✅ A/Dセグメント ({len(X_train_ad):,}件) の予測完了 (model_a_new使用)。")
else:
    print("   - ℹ️ A/Dセグメントの予測対象件数が0件のためスキップしました。")


# 3. Eセグメントの予測 (y_pred_e_train を使用)
mask_e_train = df_segmentation.loc[train_indices, 'Segment_Type'] == 'E:高誤差ノイズ'
X_train_e = X_train_full[mask_e_train]

if len(X_train_e) > 0:
    # 🚨 NNで更新された y_pred_e_train を使用
    y_pred_e_log = y_pred_e_train.loc[X_train_e.index] 
    y_pred_train_final.loc[X_train_e.index] = y_pred_e_log
    print(f"   - ✅ Eセグメント ({len(X_train_e):,}件) の予測完了 (NNモデル使用)。")
else:
    print("   - ℹ️ Eセグメントの予測対象件数が0件のためスキップしました。")


# 4. B/C セグメントの予測 (平均値代入)
y_pred_ade_combined = y_pred_train_final.dropna()
average_pred_log = y_pred_ade_combined.mean()

# B/C (未予測の NA 部分) に平均値を代入
missing_indices_train = y_pred_train_final[y_pred_train_final.isna()].index
y_pred_train_final.loc[missing_indices_train] = average_pred_log
print(f"   - 💡 B/C (未予測 {len(missing_indices_train):,}件) に平均値 {average_pred_log:.6f} を代入完了。")


# 5. 最終MAEの計算 (訓練データ評価)
y_train_full_sorted = Y_train_full.loc[y_pred_train_final.index]
final_mae_train = mean_absolute_error(y_train_full_sorted, y_pred_train_final)

print("-" * 50)
print(f"   - 💡 最終統合件数 (訓練データ): {len(y_pred_train_final):,}件")
print(f"   - 💡 最終 MAE (訓練データ全体): {final_mae_train:.6f}")
print("-" * 50)


--- Step 3-2: A/D, E への訓練データ適用と予測統合 (最終評価) ---
   - ✅ A/Dセグメント (546,574件) の予測完了 (model_a_new使用)。
   - ✅ Eセグメント (5,218件) の予測完了 (NNモデル使用)。
   - 💡 B/C (未予測 14,888件) に平均値 7.238565 を代入完了。
--------------------------------------------------
   - 💡 最終統合件数 (訓練データ): 566,680件
   - 💡 最終 MAE (訓練データ全体): 0.065091
--------------------------------------------------


In [83]:
df.columns

Index(['市区町村コード', '都道府県名', '市区町村名', '地区名', '最寄駅：名称', '最寄駅：距離（分）', '間取り',
       '面積（㎡）', '建築年', '建物の構造', '用途', '今後の利用目的', '都市計画', '建ぺい率（％）', '容積率（％）',
       '取引時点', '取引価格（総額）_log', '取引時点_年', '建築年_西暦', '築年数_欠損', '取引時点での築年数',
       '取引の事情等_調停・競売等', '取引の事情等_関係者間取引', '取引の事情等_その他', '改装_改装済', '改装_未改装',
       '間取り_grouped_その他', '間取り_grouped_オープンフロア', '間取り_grouped_欠損値',
       '間取り_grouped_１ＤＫ', '間取り_grouped_１Ｋ', '間取り_grouped_１ＬＤＫ',
       '間取り_grouped_１Ｒ', '間取り_grouped_２ＤＫ', '間取り_grouped_２Ｋ',
       '間取り_grouped_２ＬＤＫ', '間取り_grouped_２ＬＤＫ＋Ｓ', '間取り_grouped_３ＤＫ',
       '間取り_grouped_３ＬＤＫ', '間取り_grouped_４ＤＫ', '間取り_grouped_４ＬＤＫ', '都市計画_高価格帯',
       '都市計画_中価格帯', '都市計画_低価格帯', '人口密度', '市区町村人口密度', '犯罪発生率', '築年数_2乗',
       '築年数_3乗', '築年数_log', '面積_log', '面積_平方根', '築年数×面積', '築年数×駅距離',
       '築年数×建ぺい率', '築年数×容積率', '築年数×人口密度', '面積×駅距離', '面積×建ぺい率', '面積×容積率',
       '面積×人口密度', '建築可能性', '容積率_建ぺい率比', '建ぺい率_2乗', '容積率_2乗', '駅距離_逆数',
       '駅距離_log', '駅距離_2乗', '駅距離×建ぺい率', '駅距離×容積率', '人口密度_log', '市区町村人口密度_l

In [80]:
import joblib
import os
import numpy as np
from tensorflow.keras.models import save_model

# ディレクトリの定義 (必要に応じて訓練側で定義されているパスに合わせる)
MODEL_SAVE_DIR = '../models/'  # 訓練済みモデルを保存するディレクトリ
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

# -----------------
# 1. A/Dセグメント XGBoost モデルの保存
# -----------------
# model_a_new が最終的な A/D モデルと仮定
joblib.dump(model_a_new, os.path.join(MODEL_SAVE_DIR, 'model_a_new.pkl'))
print(f"✅ A/D XGBoost モデルを {MODEL_SAVE_DIR} に保存しました。")

# -----------------
# 2. Eセグメント NN モデルの保存 (Keras形式)
# -----------------
# model_e_nn が最終的な E モデルと仮定
save_model(model_e_nn, os.path.join(MODEL_SAVE_DIR, 'model_e_nn.keras'))
print(f"✅ E NN モデルを {MODEL_SAVE_DIR} に保存しました。")

# -----------------
# 3. B/Cセグメントの予測平均値 (対数) の計算と保存
# -----------------
# 訓練データ全体に対する最終予測 (y_pred_train_final) の統合平均値を使用
# 最終統合MAE計算後の y_pred_train_final から平均値を計算
if 'y_pred_train_final' in locals():
    # 平均値代入に使用した平均値 (対数) を取得
    # B/Cの平均値は、A/D/Eの予測値全体の平均値を使用しました
    average_pred_log = y_pred_train_final.dropna().mean()
else:
    # 🚨 y_pred_train_final が定義されていない場合、訓練データ全体で予測し直す必要があります
    # 簡易的に、訓練データのYの平均値を使用 (これは厳密ではない)
    average_pred_log = Y_train_full.mean()
    print("⚠️ y_pred_train_final が見つからないため、Y_train_full の平均値を使用しました。")


# B/C平均値と特徴量リストを一つの辞書にまとめて保存
prediction_params = {
    'bc_average_log': average_pred_log,
    'numeric_features': NUMERIC_FEATURES,
    'categorical_features': CATEGORICAL_FEATURES
}
joblib.dump(prediction_params, os.path.join(MODEL_SAVE_DIR, 'prediction_params.pkl'))
print(f"✅ B/C 平均値 ({average_pred_log:.6f}) および特徴量リストを保存しました。")

✅ A/D XGBoost モデルを ../models/ に保存しました。
✅ E NN モデルを ../models/ に保存しました。
✅ B/C 平均値 (7.238565) および特徴量リストを保存しました。


In [81]:
import joblib
import os

MODEL_SAVE_DIR = '../models/'
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

# 訓練済みの classifier_ae変数を使って保存
joblib.dump(classifier_ae, os.path.join(MODEL_SAVE_DIR, 'classifier_noise.pkl'))
print("✅ ノイズ分類モデル (classifier_ae) を保存しました。")

✅ ノイズ分類モデル (classifier_ae) を保存しました。
